# User Guide
### Jeremy Ball, Omer Bowman, Richard Ngai


In [66]:
# Import Libraries
import numpy as np
import time
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# Connect to CoppeliaSim

The following code connects to an open session of CoppeliaSim. It will then load in the template MTB SCARA robot found in CoppeliaSim's model library.

In [67]:
# Connect to CoppeliaSim
print('Connecting...')
client = RemoteAPIClient('localhost', 23000)
sim = client.require('sim')
print('Connected')
sim.setBoolParam(sim.boolparam_realtime_simulation, True)


Connecting...
Connected


1

# Load and configure the MTB SCARA

The user can choose to run this block to have the SCARA load in automatically. The following code
loads the MTB SCARA robot arm from CoppeliaSim's default library. The code also removes the default
script located within the heirarchy.

If the user is unable to run the code, they can manually place the MTB robot from the Model Browser
under robots>non-mobile>MTB robot. If manually added, the user must then delete 'Script' found under
the MTB parent.

In [68]:
# Load robot
robot = sim.loadModel(r'C:\Program Files\CoppeliaRobotics\CoppeliaSimEdu\models\robots\non-mobile\MTB robot.ttm')
print('Robot loaded')

# Remove default script
script = sim.getScript(sim.scripttype_childscript, robot)
if script != -1:
    sim.removeObjects([script])

Robot loaded


# Assign Joints

The following code block assigns joints q1, q2, q3, and q4 based on the MTB's structure.

In [69]:
# Assign MTB joints
q1 = sim.getObject('/MTB/axis')
q2 = sim.getObject('/MTB/axis/link/axis')
q3 = sim.getObject('/MTB/axis/link/axis/link/axis')
q4 = sim.getObject('/MTB/axis/link/axis/link/axis/axis')
print('Joints assigned')

# Get positions of the joints
o1 = np.array(sim.getObjectPosition(q1, sim.handle_world))
o2 = np.array(sim.getObjectPosition(q2, sim.handle_world))
o3 = np.array(sim.getObjectPosition(q3, sim.handle_world))

# Compute link lengths
l1 = np.linalg.norm(o2-o1)
l2 = np.linalg.norm(o3-o2)


Joints assigned


# Set up the Box class

In [70]:
# Define a Box class that will be used to create boxes in the simulation

class Box:
    def __init__(self, sim, position, size=[0.1, 0.1, 0.1], velocity=[0.1, 0, 0]):
        self.sim = sim
        self.size = size
        self.velocity = np.array(velocity)
        self.attached = False

        self.handle = sim.createPrimitiveShape(
            sim.primitiveshape_cuboid,
            size,
            0
        )

        sim.setObjectPosition(self.handle, list(position), sim.handle_world)
        sim.setObjectInt32Param(self.handle, sim.shapeintparam_static, 1)
        sim.setShapeMass(self.handle, 0.1)
        sim.setShapeColor(self.handle, None, sim.colorcomponent_ambient_diffuse, [0.65, 0.65, 1])
    
    def update(self, dt):
        if not self.attached:
            pos = np.array(self.sim.getObjectPosition(self.handle, self.sim.handle_world))
            new_pos = pos + self.velocity * dt
            self.sim.setObjectPosition(self.handle, list(new_pos), self.sim.handle_world)
    
    def attach(self, tool_handle):
        self.sim.setObjectParent(self.handle, tool_handle, True)
        self.attached = True

    def detach(self):
        self.sim.setObjectParent(self.handle, -1, True)
        self.attached = False

# Run Simulation

The following code runs the simulation in CoppeliaSim.

In [71]:

box = Box(sim, position=[0.2, -0.3, 0.05])
# Start the simulation
print('Starting simulation...')
sim.setStepping(True)
sim.startSimulation()




#~~~~~~~~~~~~~~~~~~~~~~~~~
# TODO SIMULATION



dt = sim.getSimulationTimeStep()

for i in range(200):
    box.update(dt)
    sim.step()

u1 = np.radians(-30)
u2 = np.radians(-20)
u3 = 0.0

sim.setJointTargetPosition(q1, u1)
sim.setJointTargetPosition(q2, u2)
sim.setJointTargetPosition(q3, u3)

for _ in range(60):
    sim.step()

u1 = np.radians(30)
u2 = np.radians(-20)
u3 = 0.10

sim.setJointTargetPosition(q1, u1)
sim.setJointTargetPosition(q2, u2)
sim.setJointTargetPosition(q3, u3)

for _ in range(60):
    sim.step()
#~~~~~~~~~~~~~~~~~~~~~~~~~





# Stop the simulation
sim.stopSimulation()
sim.setStepping(False)
print('Stopped')

# Remove MTB model
sim.removeModel(robot)

Starting simulation...
Stopped


21